In [ ]:
# try:
#     from google.colab import drive
#     drive.mount("/content/drive")
# except ImportError:
#     pass  # not running on Colab

DATA_DIR = "/content/drive/MyDrive/KDU/Thesis/Dataset"   # folder with tram_train/dev/test.tsv
OUT_ROOT = "/content/drive/MyDrive/KDU/Thesis/runs"      # checkpoints + metrics are written here

In [ ]:
import ast, json, os, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup)
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, precision_score, recall_score, hamming_loss

# Exp 0 — TRAM multi-label ATT&CK classification (RoBERTa / SecBERT)

Consolidated version of the notebook. Changes from the previous one:
- one `train_model` (no `_fixed` duplicate); `text1` is renamed to `text` once, at load time
- `pos_weight` is capped (default 20) instead of reaching ~3467x for singleton classes
- early stopping on dev micro-F1 (`epochs` is now an upper bound)
- macro-F1 is computed only over classes present in the split being scored
- rows where no class crosses the threshold fall back to the top-scoring class
- extra metrics: subset accuracy and Hamming loss
- outputs saved to Google Drive so a Colab disconnect doesn't lose the checkpoints

In [ ]:
def load_split(name):
    df = pd.read_csv(f"{DATA_DIR}/tram_{name}.tsv", sep="\t").rename(columns={"text1": "text"})
    df["y"] = df["labels"].map(lambda x: sorted(set(ast.literal_eval(x))))  # parse + dedupe labels
    return df[["text", "y"]]

train, dev, test = (load_split(s) for s in ("train", "dev", "test"))
train = train.drop_duplicates(subset=["text"]).reset_index(drop=True)   # train only; never shrink dev/test

mlb = MultiLabelBinarizer().fit(train["y"])          # label space = classes seen in train
Y_train, Y_dev, Y_test = (mlb.transform(d["y"]) for d in (train, dev, test))
# sklearn warns about dev/test labels that never occur in train; they are unlearnable, so dropping them is expected
print(f"train={len(train)} dev={len(dev)} test={len(test)} classes={len(mlb.classes_)}")

train=3468 dev=604 test=725 classes=180


/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_label.py:909: UserWarning: unknown class(es) ['T1055.011', 'T1216', 'T1547.005', 'T1569.001'] will be ignored
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_label.py:909: UserWarning: unknown class(es) ['T1011', 'T1134.005', 'T1218.002', 'T1485', 'T1547.005', 'T1548.001', 'T1548.003', 'T1552.004', 'T1556.002', 'T1574.010'] will be ignored
  warnings.warn(


In [ ]:
# print(Y_train[:5])
# print(Y_test[:5])
# print(Y_dev[:5])

In [ ]:
def compute_pos_weight(Y, cap=20.0):
    pos = np.clip(Y.sum(axis=0), 1, None)
    return np.clip((Y.shape[0] - pos) / pos, None, cap)

def best_global_threshold(y_true, probs, grid=np.arange(0.1, 0.9, 0.05)):
    scores = [f1_score(y_true, (probs >= t).astype(int), average="micro", zero_division=0) for t in grid]
    return float(grid[int(np.argmax(scores))])

def best_per_class_thresholds(y_true, probs, grid=np.arange(0.05, 0.95, 0.05)):
    th = np.full(y_true.shape[1], 0.5)
    for c in range(y_true.shape[1]):
        if y_true[:, c].sum() == 0:
            continue                                  # class absent from dev -> keep default
        s = [f1_score(y_true[:, c], (probs[:, c] >= t).astype(int), zero_division=0) for t in grid]
        th[c] = grid[int(np.argmax(s))]
    return th

def apply_thresholds(probs, thresholds):
    preds = (probs >= thresholds).astype(int)         # thresholds: scalar or per-class array
    for i in np.where(preds.sum(axis=1) == 0)[0]:     # never predict "no technique"
        preds[i, probs[i].argmax()] = 1
    return preds

def evaluate(y_true, y_pred):
    present = y_true.sum(axis=0) > 0                  # macro-F1 only over classes that occur in this split
    return {
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_f1": f1_score(y_true[:, present], y_pred[:, present], average="macro", zero_division=0),
        "micro_precision": precision_score(y_true, y_pred, average="micro", zero_division=0),
        "micro_recall": recall_score(y_true, y_pred, average="micro", zero_division=0),
        "subset_accuracy": float((y_true == y_pred).all(axis=1).mean()),
        "hamming_loss": hamming_loss(y_true, y_pred),
        "n_classes_present": int(present.sum()),
    }

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

In [ ]:
class TramDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts, self.labels = list(texts), labels
        self.tokenizer, self.max_length = tokenizer, max_length
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, i):
        enc = self.tokenizer(self.texts[i], truncation=True, max_length=self.max_length,
                             padding="max_length", return_tensors="pt")
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[i], dtype=torch.float)
        return item

def run_epoch(model, loader, loss_fn, device, optimizer=None, scheduler=None, desc=""):
    train_mode = optimizer is not None
    model.train() if train_mode else model.eval()
    total, probs, labs = 0.0, [], []
    bar = tqdm(loader, desc=desc, leave=False)
    for batch in bar:
        batch = {k: v.to(device) for k, v in batch.items()}
        y = batch.pop("labels")
        with torch.set_grad_enabled(train_mode):
            logits = model(**batch).logits
            loss = loss_fn(logits, y)
            if train_mode:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
        total += loss.item() * y.size(0)
        probs.append(torch.sigmoid(logits).detach().cpu().numpy())
        labs.append(y.detach().cpu().numpy())
        bar.set_postfix(loss=f"{loss.item():.4f}")
    return total / len(loader.dataset), np.concatenate(probs), np.concatenate(labs)

In [ ]:
def train_model(model_name, output_dir, seed=42, epochs=15, patience=3, batch_size=16, lr=2e-5,
                max_length=128, warmup_ratio=0.1, use_pos_weight=True, pos_weight_cap=20.0,
                per_class_threshold=False):
    set_seed(seed)
    os.makedirs(output_dir, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    num_classes = len(mlb.classes_)
    print(f"[{model_name} seed={seed}] device={device} train={len(train)} dev={len(dev)} "
          f"test={len(test)} classes={num_classes}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    mk = lambda df, Y: TramDataset(df["text"], Y, tokenizer, max_length)
    train_loader = DataLoader(mk(train, Y_train), batch_size=batch_size, shuffle=True)
    dev_loader = DataLoader(mk(dev, Y_dev), batch_size=batch_size * 2)
    test_loader = DataLoader(mk(test, Y_test), batch_size=batch_size * 2)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=num_classes, problem_type="multi_label_classification").to(device)

    pos_weight = None
    if use_pos_weight:
        pos_weight = torch.tensor(compute_pos_weight(Y_train, cap=pos_weight_cap),
                                  dtype=torch.float, device=device)
        print(f"  pos_weight: min={pos_weight.min():.1f} max={pos_weight.max():.1f}")
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    total_steps = len(train_loader) * epochs
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, int(total_steps * warmup_ratio), total_steps)

    # train, keep the best checkpoint by dev micro-F1, stop early
    best_f1, best_state, bad_epochs = -1.0, None, 0
    for epoch in range(1, epochs + 1):
        tr_loss, _, _ = run_epoch(model, train_loader, loss_fn, device, optimizer, scheduler,
                                  desc=f"Epoch {epoch} [train]")
        dv_loss, dv_probs, dv_labels = run_epoch(model, dev_loader, loss_fn, device,
                                                 desc=f"Epoch {epoch} [dev]")
        m = evaluate(dv_labels, apply_thresholds(dv_probs, best_global_threshold(dv_labels, dv_probs)))
        print(f"  epoch {epoch}: train_loss={tr_loss:.4f} dev_loss={dv_loss:.4f} "
              f"dev_micro_f1={m['micro_f1']:.4f} dev_macro_f1={m['macro_f1']:.4f}")
        if m["micro_f1"] > best_f1:
            best_f1, bad_epochs = m["micro_f1"], 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f"  early stop: no dev improvement for {patience} epochs")
                break
    model.load_state_dict(best_state)

    # thresholds are tuned on dev with the best checkpoint; test is scored once
    _, dv_probs, dv_labels = run_epoch(model, dev_loader, loss_fn, device, desc="Threshold tuning")
    thresholds = (best_per_class_thresholds(dv_labels, dv_probs) if per_class_threshold
                  else best_global_threshold(dv_labels, dv_probs))
    _, te_probs, te_labels = run_epoch(model, test_loader, loss_fn, device, desc="Test")
    te_preds = apply_thresholds(te_probs, thresholds)
    metrics = evaluate(te_labels, te_preds)
    print(f"  TEST: { {k: round(v, 4) if isinstance(v, float) else v for k, v in metrics.items()} }")

    # everything the SHAP phase and the results table need
    model.save_pretrained(output_dir); tokenizer.save_pretrained(output_dir)
    np.save(f"{output_dir}/mlb_classes.npy", mlb.classes_)
    np.save(f"{output_dir}/test_probs.npy", te_probs)
    np.save(f"{output_dir}/test_preds.npy", te_preds)
    np.save(f"{output_dir}/test_labels.npy", te_labels)
    json.dump(np.asarray(thresholds).tolist(), open(f"{output_dir}/thresholds.json", "w"))
    json.dump({"model_name": model_name, "seed": seed, "dev_best_micro_f1": best_f1, "test": metrics,
               "config": dict(lr=lr, batch_size=batch_size, max_length=max_length,
                              use_pos_weight=use_pos_weight, pos_weight_cap=pos_weight_cap,
                              per_class_threshold=per_class_threshold)},
              open(f"{output_dir}/metrics.json", "w"), indent=2)
    print(f"  saved to {output_dir}")
    return metrics

## Run one model, one seed (sanity check first)

In [ ]:
train_model("roberta-base", f"{OUT_ROOT}/roberta_s42", seed=42)

[roberta-base seed=42] device=cuda train=3468 dev=604 test=725 classes=180


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  pos_weight: min=19.4 max=20.0


Epoch 1 [train]:   0%|          | 0/217 [00:00<?, ?it/s]

Epoch 1 [dev]:   0%|          | 0/19 [00:00<?, ?it/s]

... (7 outputs left out)

Epoch 4 [train]:   0%|          | 0/217 [00:00<?, ?it/s]

Epoch 4 [dev]:   0%|          | 0/19 [00:00<?, ?it/s]

  epoch 4: train_loss=0.2315 dev_loss=0.2223 dev_micro_f1=0.4532 dev_macro_f1=0.1763


Epoch 5 [train]:   0%|          | 0/217 [00:00<?, ?it/s]

## Ablations worth running once (RoBERTa, seed 42)
Compare dev/test micro-F1 and macro-F1 to decide the loss setup you'll use for every model.

In [ ]:
# A/B: no class weighting at all
# train_model("roberta-base", f"{OUT_ROOT}/roberta_nopw_s42", seed=42, use_pos_weight=False)

# A/B: weaker / stronger cap
# train_model("roberta-base", f"{OUT_ROOT}/roberta_cap5_s42",  seed=42, pos_weight_cap=5.0)
# train_model("roberta-base", f"{OUT_ROOT}/roberta_cap50_s42", seed=42, pos_weight_cap=50.0)

# per-class thresholds (often helps macro-F1)
# train_model("roberta-base", f"{OUT_ROOT}/roberta_pcth_s42", seed=42, per_class_threshold=True)

## Final runs: 3 seeds per model, then mean ± std
Run this once the loss setup is decided. Confirm the exact SecBERT checkpoint ID on the Hugging Face Hub first.

In [ ]:
MODELS = {"secbert": "jackaduma/SecBERT", "roberta": "roberta-base"}   # <- verify the SecBERT ID
SEEDS = [42, 43, 44]

for tag, name in MODELS.items():
    for s in SEEDS:
        out = f"{OUT_ROOT}/{tag}_s{s}"
        if os.path.exists(f"{out}/metrics.json"):
            print(f"skip {out} (already done)")       # lets you resume after a Colab disconnect
            continue
        train_model(name, out, seed=s)

## Results table: mean ± std over seeds

In [ ]:
rows = []
for tag in MODELS:
    ms = [json.load(open(f"{OUT_ROOT}/{tag}_s{s}/metrics.json"))["test"] for s in SEEDS
          if os.path.exists(f"{OUT_ROOT}/{tag}_s{s}/metrics.json")]
    if not ms:
        continue
    row = {"model": tag, "n_seeds": len(ms)}
    for k in ("micro_f1", "macro_f1", "micro_precision", "micro_recall", "subset_accuracy"):
        v = np.array([m[k] for m in ms])
        row[k] = f"{v.mean():.3f} ± {v.std():.3f}"
    rows.append(row)
pd.DataFrame(rows)

## Optional: TF-IDF + SVM baseline (CPU, seconds)
Uses the same splits, label space and metrics, so it is directly comparable. Dev is used only to pick the threshold.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import LinearSVC

vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True).fit(train["text"])   # fit on train only
Xtr, Xdv, Xte = (vec.transform(d["text"]) for d in (train, dev, test))

# LinearSVC needs at least one positive per class column; all classes come from train, so this holds
svm = OneVsRestClassifier(LinearSVC(C=1.0)).fit(Xtr, Y_train)
sc_dv, sc_te = svm.decision_function(Xdv), svm.decision_function(Xte)

grid = np.arange(-0.6, 0.31, 0.05)                       # decision scores are not probabilities
best_t = grid[int(np.argmax([f1_score(Y_dev, (sc_dv >= t).astype(int), average="micro", zero_division=0)
                             for t in grid]))]
print("SVM dev :", {k: round(v, 3) for k, v in evaluate(Y_dev, apply_thresholds(sc_dv, best_t)).items()})
print("SVM test:", {k: round(v, 3) for k, v in evaluate(Y_test, apply_thresholds(sc_te, best_t)).items()})

# Exp 0 — TRAM multi-label ATT&CK classification (RoBERTa / SecBERT)

Consolidated version of the notebook. Changes from the previous one:
- one `train_model` (no `_fixed` duplicate); `text1` is renamed to `text` once, at load time
- `pos_weight` is capped (default 20) instead of reaching ~3467x for singleton classes
- early stopping on dev micro-F1 (`epochs` is now an upper bound)
- macro-F1 is computed only over classes present in the split being scored
- rows where no class crosses the threshold fall back to the top-scoring class
- extra metrics: subset accuracy and Hamming loss
- outputs saved to Google Drive so a Colab disconnect doesn't lose the checkpoints

In [ ]:
def train_model(model_name, output_dir, seed=42, epochs=15, patience=3, batch_size=16, lr=2e-5,
                max_length=128, warmup_ratio=0.1, use_pos_weight=True, pos_weight_cap=20.0,
                per_class_threshold=False):
    set_seed(seed)
    os.makedirs(output_dir, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    num_classes = len(mlb.classes_)
    print(f"[{model_name} seed={seed}] device={device} train={len(train)} dev={len(dev)} "
          f"test={len(test)} classes={num_classes}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    mk = lambda df, Y: TramDataset(df["text"], Y, tokenizer, max_length)
    train_loader = DataLoader(mk(train, Y_train), batch_size=batch_size, shuffle=True)
    dev_loader = DataLoader(mk(dev, Y_dev), batch_size=batch_size * 2)
    test_loader = DataLoader(mk(test, Y_test), batch_size=batch_size * 2)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=num_classes, problem_type="multi_label_classification").to(device)

    pos_weight = None
    if use_pos_weight:
        pos_weight = torch.tensor(compute_pos_weight(Y_train, cap=pos_weight_cap),
                                  dtype=torch.float, device=device)
        print(f"  pos_weight: min={pos_weight.min():.1f} max={pos_weight.max():.1f}")
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    total_steps = len(train_loader) * epochs
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, int(total_steps * warmup_ratio), total_steps)

    # train, keep the best checkpoint by dev micro-F1, stop early
    best_f1, best_state, bad_epochs = -1.0, None, 0
    for epoch in range(1, epochs + 1):
        tr_loss, _, _ = run_epoch(model, train_loader, loss_fn, device, optimizer, scheduler,
                                  desc=f"Epoch {epoch} [train]")
        dv_loss, dv_probs, dv_labels = run_epoch(model, dev_loader, loss_fn, device,
                                                 desc=f"Epoch {epoch} [dev]")
        m = evaluate(dv_labels, apply_thresholds(dv_probs, best_global_threshold(dv_labels, dv_probs)))
        print(f"  epoch {epoch}: train_loss={tr_loss:.4f} dev_loss={dv_loss:.4f} "
              f"dev_micro_f1={m['micro_f1']:.4f} dev_macro_f1={m['macro_f1']:.4f}")
        if m["micro_f1"] > best_f1:
            best_f1, bad_epochs = m["micro_f1"], 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f"  early stop: no dev improvement for {patience} epochs")
                break
    model.load_state_dict(best_state)

    # thresholds are tuned on dev with the best checkpoint; test is scored once
    _, dv_probs, dv_labels = run_epoch(model, dev_loader, loss_fn, device, desc="Threshold tuning")
    thresholds = (best_per_class_thresholds(dv_labels, dv_probs) if per_class_threshold
                  else best_global_threshold(dv_labels, dv_probs))
    _, te_probs, te_labels = run_epoch(model, test_loader, loss_fn, device, desc="Test")
    te_preds = apply_thresholds(te_probs, thresholds)
    metrics = evaluate(te_labels, te_preds)
    print(f"  TEST: { {k: round(v, 4) if isinstance(v, float) else v for k, v in metrics.items()} }")

    # everything the SHAP phase and the results table need
    model.save_pretrained(output_dir); tokenizer.save_pretrained(output_dir)
    np.save(f"{output_dir}/mlb_classes.npy", mlb.classes_)
    np.save(f"{output_dir}/test_probs.npy", te_probs)
    np.save(f"{output_dir}/test_preds.npy", te_preds)
    np.save(f"{output_dir}/test_labels.npy", te_labels)
    json.dump(np.asarray(thresholds).tolist(), open(f"{output_dir}/thresholds.json", "w"))
    json.dump({"model_name": model_name, "seed": seed, "dev_best_micro_f1": best_f1, "test": metrics,
               "config": dict(lr=lr, batch_size=batch_size, max_length=max_length,
                              use_pos_weight=use_pos_weight, pos_weight_cap=pos_weight_cap,
                              per_class_threshold=per_class_threshold)},
              open(f"{output_dir}/metrics.json", "w"), indent=2)
    print(f"  saved to {output_dir}")
    return metrics

## Run one model, one seed (sanity check first)

In [ ]:
train_model("roberta-base", f"{OUT_ROOT}/roberta_s42", seed=42)

## Ablations worth running once (RoBERTa, seed 42)
Compare dev/test micro-F1 and macro-F1 to decide the loss setup you'll use for every model.

In [ ]:
# A/B: no class weighting at all
# train_model("roberta-base", f"{OUT_ROOT}/roberta_nopw_s42", seed=42, use_pos_weight=False)

# A/B: weaker / stronger cap
# train_model("roberta-base", f"{OUT_ROOT}/roberta_cap5_s42",  seed=42, pos_weight_cap=5.0)
# train_model("roberta-base", f"{OUT_ROOT}/roberta_cap50_s42", seed=42, pos_weight_cap=50.0)

# per-class thresholds (often helps macro-F1)
# train_model("roberta-base", f"{OUT_ROOT}/roberta_pcth_s42", seed=42, per_class_threshold=True)

## Final runs: 3 seeds per model, then mean ± std
Run this once the loss setup is decided. Confirm the exact SecBERT checkpoint ID on the Hugging Face Hub first.

In [ ]:
MODELS = {"secbert": "jackaduma/SecBERT", "roberta": "roberta-base"}   # <- verify the SecBERT ID
SEEDS = [42, 43, 44]

for tag, name in MODELS.items():
    for s in SEEDS:
        out = f"{OUT_ROOT}/{tag}_s{s}"
        if os.path.exists(f"{out}/metrics.json"):
            print(f"skip {out} (already done)")       # lets you resume after a Colab disconnect
            continue
        train_model(name, out, seed=s)

In [2]:
# train_model(MODELS['secbert'], f"{OUT_ROOT}/secbert_s42", seed=42)
# Removed this line as it's redundant. Please execute cell bf0566c2 after defining train_model in cell 39c2afa8.

## Results table: mean ± std over seeds

In [ ]:
rows = []
for tag in MODELS:
    ms = [json.load(open(f"{OUT_ROOT}/{tag}_s{s}/metrics.json"))["test"] for s in SEEDS
          if os.path.exists(f"{OUT_ROOT}/{tag}_s{s}/metrics.json")]
    if not ms:
        continue
    row = {"model": tag, "n_seeds": len(ms)}
    for k in ("micro_f1", "macro_f1", "micro_precision", "micro_recall", "subset_accuracy"):
        v = np.array([m[k] for m in ms])
        row[k] = f"{v.mean():.3f} ± {v.std():.3f}"
    rows.append(row)
pd.DataFrame(rows)

## Optional: TF-IDF + SVM baseline (CPU, seconds)
Uses the same splits, label space and metrics, so it is directly comparable. Dev is used only to pick the threshold.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import LinearSVC

vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True).fit(train["text"])   # fit on train only
Xtr, Xdv, Xte = (vec.transform(d["text"]) for d in (train, dev, test))

# LinearSVC needs at least one positive per class column; all classes come from train, so this holds
svm = OneVsRestClassifier(LinearSVC(C=1.0)).fit(Xtr, Y_train)
sc_dv, sc_te = svm.decision_function(Xdv), svm.decision_function(Xte)

grid = np.arange(-0.6, 0.31, 0.05)                       # decision scores are not probabilities
best_t = grid[int(np.argmax([f1_score(Y_dev, (sc_dv >= t).astype(int), average="micro", zero_division=0)
                             for t in grid]))]
print("SVM dev :", {k: round(v, 3) for k, v in evaluate(Y_dev, apply_thresholds(sc_dv, best_t)).items()})
print("SVM test:", {k: round(v, 3) for k, v in evaluate(Y_test, apply_thresholds(sc_te, best_t)).items()})

# Task
Execute the training loop over multiple seeds for the SecBERT and RoBERTa models using the predefined `train_model` infrastructure with the datasets in `"/content/drive/MyDrive/KDU/Thesis/Dataset"`, then compile and present the final results table showing the mean and standard deviation of key performance metrics (such as micro-F1, macro-F1, micro-precision, micro-recall, and subset accuracy) saved under `"/content/drive/MyDrive/KDU/Thesis/runs"`.

## Define Model Training Infrastructure

### Subtask:
Register the necessary training infrastructure, datasets, and helper functions in the current session.


**Reasoning**:
Load and run the cell containing model training setup, dataset helpers, and dependencies to initialize the training infrastructure.



In [3]:
import ast, json, os, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup)
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, precision_score, recall_score, hamming_loss

# Define the paths
DATA_DIR = "/content/drive/MyDrive/KDU/Thesis/Dataset"
OUT_ROOT = "/content/drive/MyDrive/KDU/Thesis/runs"

# Helper to load data splits
def load_split(name):
    df = pd.read_csv(f"{DATA_DIR}/tram_{name}.tsv", sep="\t").rename(columns={"text1": "text"})
    df["y"] = df["labels"].map(lambda x: sorted(set(ast.literal_eval(x))))
    return df[["text", "y"]]

try:
    train, dev, test = (load_split(s) for s in ("train", "dev", "test"))
    train = train.drop_duplicates(subset=["text"]).reset_index(drop=True)
    mlb = MultiLabelBinarizer().fit(train["y"])
    Y_train, Y_dev, Y_test = (mlb.transform(d["y"]) for d in (train, dev, test))
    print(f"Dataset loaded: train={len(train)} dev={len(dev)} test={len(test)} classes={len(mlb.classes_)}")
except Exception as e:
    print(f"Warning: Could not load data directly from Google Drive. Ensure Drive is mounted. Error: {e}")

# Define Dataset and running utilities
class TramDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts, self.labels = list(texts), labels
        self.tokenizer, self.max_length = tokenizer, max_length
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, i):
        enc = self.tokenizer(self.texts[i], truncation=True, max_length=self.max_length,
                             padding="max_length", return_tensors="pt")
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[i], dtype=torch.float)
        return item

def run_epoch(model, loader, loss_fn, device, optimizer=None, scheduler=None, desc=""):
    train_mode = optimizer is not None
    model.train() if train_mode else model.eval()
    total, probs, labs = 0.0, [], []
    bar = tqdm(loader, desc=desc, leave=False)
    for batch in bar:
        batch = {k: v.to(device) for k, v in batch.items()}
        y = batch.pop("labels")
        with torch.set_grad_enabled(train_mode):
            logits = model(**batch).logits
            loss = loss_fn(logits, y)
            if train_mode:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
        total += loss.item() * y.size(0)
        probs.append(torch.sigmoid(logits).detach().cpu().numpy())
        labs.append(y.detach().cpu().numpy())
        bar.set_postfix(loss=f"{loss.item():.4f}")
    return total / len(loader.dataset), np.concatenate(probs), np.concatenate(labs)

def compute_pos_weight(Y, cap=20.0):
    pos = np.clip(Y.sum(axis=0), 1, None)
    return np.clip((Y.shape[0] - pos) / pos, None, cap)

def best_global_threshold(y_true, probs, grid=np.arange(0.1, 0.9, 0.05)):
    scores = [f1_score(y_true, (probs >= t).astype(int), average="micro", zero_division=0) for t in grid]
    return float(grid[int(np.argmax(scores))])

def best_per_class_thresholds(y_true, probs, grid=np.arange(0.05, 0.95, 0.05)):
    th = np.full(y_true.shape[1], 0.5)
    for c in range(y_true.shape[1]):
        if y_true[:, c].sum() == 0:
            continue
        s = [f1_score(y_true[:, c], (probs[:, c] >= t).astype(int), zero_division=0) for t in grid]
        th[c] = grid[int(np.argmax(s))]
    return th

def apply_thresholds(probs, thresholds):
    preds = (probs >= thresholds).astype(int)
    for i in np.where(preds.sum(axis=1) == 0)[0]:
        preds[i, probs[i].argmax()] = 1
    return preds

def evaluate(y_true, y_pred):
    present = y_true.sum(axis=0) > 0
    return {
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_f1": f1_score(y_true[:, present], y_pred[:, present], average="macro", zero_division=0),
        "micro_precision": precision_score(y_true, y_pred, average="micro", zero_division=0),
        "micro_recall": recall_score(y_true, y_pred, average="micro", zero_division=0),
        "subset_accuracy": float((y_true == y_pred).all(axis=1).mean()),
        "hamming_loss": hamming_loss(y_true, y_pred),
        "n_classes_present": int(present.sum()),
    }

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

print("Training helper infrastructures and configurations are successfully registered in the kernel.")

Dataset loaded: train=3468 dev=604 test=725 classes=180
Training helper infrastructures and configurations are successfully registered in the kernel.


/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_label.py:909: UserWarning: unknown class(es) ['T1055.011', 'T1216', 'T1547.005', 'T1569.001'] will be ignored
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_label.py:909: UserWarning: unknown class(es) ['T1011', 'T1134.005', 'T1218.002', 'T1485', 'T1547.005', 'T1548.001', 'T1548.003', 'T1552.004', 'T1556.002', 'T1574.010'] will be ignored
  warnings.warn(


**Reasoning**:
We need to compile and register the train_model function in the notebook environment using clean, unescaped string syntax to avoid any further SyntaxErrors during evaluation.



In [8]:
def train_model(model_name, output_dir, seed=42, epochs=15, patience=3, batch_size=16, lr=2e-5, max_length=128, warmup_ratio=0.1, use_pos_weight=True, pos_weight_cap=20.0, per_class_threshold=False):
    set_seed(seed)
    os.makedirs(output_dir, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    num_classes = len(mlb.classes_)
    print(f"[{model_name} seed={seed}] device={device} train={len(train)} dev={len(dev)} test={len(test)} classes={num_classes}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    mk = lambda df, Y: TramDataset(df["text"], Y, tokenizer, max_length)
    train_loader = DataLoader(mk(train, Y_train), batch_size=batch_size, shuffle=True)
    dev_loader = DataLoader(mk(dev, Y_dev), batch_size=batch_size * 2)
    test_loader = DataLoader(mk(test, Y_test), batch_size=batch_size * 2)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_classes, problem_type="multi_label_classification").to(device)

    pos_weight = None
    if use_pos_weight:
        pos_weight = torch.tensor(compute_pos_weight(Y_train, cap=pos_weight_cap), dtype=torch.float, device=device)
        print(f"  pos_weight: min={pos_weight.min():.1f} max={pos_weight.max():.1f}")
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    total_steps = len(train_loader) * epochs
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * warmup_ratio), total_steps)

    best_f1, best_state, bad_epochs = -1.0, None, 0
    for epoch in range(1, epochs + 1):
        tr_loss, _, _ = run_epoch(model, train_loader, loss_fn, device, optimizer, scheduler, desc=f"Epoch {epoch} [train]")
        dv_loss, dv_probs, dv_labels = run_epoch(model, dev_loader, loss_fn, device, desc=f"Epoch {epoch} [dev]")
        m = evaluate(dv_labels, apply_thresholds(dv_probs, best_global_threshold(dv_labels, dv_probs)))
        print(f"  epoch {epoch}: train_loss={tr_loss:.4f} dev_loss={dv_loss:.4f} dev_micro_f1={m['micro_f1']:.4f} dev_macro_f1={m['macro_f1']:.4f}")
        if m["micro_f1"] > best_f1:
            best_f1, bad_epochs = m["micro_f1"], 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f"  early stop: no dev improvement for {patience} epochs")
                break
    model.load_state_dict(best_state)

    _, dv_probs, dv_labels = run_epoch(model, dev_loader, loss_fn, device, desc="Threshold tuning")
    thresholds = best_per_class_thresholds(dv_labels, dv_probs) if per_class_threshold else best_global_threshold(dv_labels, dv_probs)
    _, te_probs, te_labels = run_epoch(model, test_loader, loss_fn, device, desc="Test")
    te_preds = apply_thresholds(te_probs, thresholds)
    metrics = evaluate(te_labels, te_preds)
    print(f"  TEST: {{k: round(v, 4) if isinstance(v, float) else v for k, v in metrics.items()}}")

    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    np.save(f"{output_dir}/mlb_classes.npy", mlb.classes_)
    np.save(f"{output_dir}/test_probs.npy", te_probs)
    np.save(f"{output_dir}/test_preds.npy", te_preds)
    np.save(f"{output_dir}/test_labels.npy", te_labels)
    json.dump(np.asarray(thresholds).tolist(), open(f"{output_dir}/thresholds.json", "w"))
    json.dump({"model_name": model_name, "seed": seed, "dev_best_micro_f1": best_f1, "test": metrics, "config": dict(lr=lr, batch_size=batch_size, max_length=max_length, use_pos_weight=use_pos_weight, pos_weight_cap=pos_weight_cap, per_class_threshold=per_class_threshold)}, open(f"{output_dir}/metrics.json", "w"), indent=2)
    print(f"  saved to {output_dir}")
    return metrics

**Reasoning**:
The training infrastructure was successfully registered in the previous cell. Now we define the main train_model function in the kernel to allow subsequent model training tasks.



In [6]:
def train_model(model_name, output_dir, seed=42, epochs=15, patience=3, batch_size=16, lr=2e-5, max_length=128, warmup_ratio=0.1, use_pos_weight=True, pos_weight_cap=20.0, per_class_threshold=False):
    set_seed(seed)
    os.makedirs(output_dir, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    num_classes = len(mlb.classes_)
    print(f"[{model_name} seed={seed}] device={device} train={len(train)} dev={len(dev)} test={len(test)} classes={num_classes}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    mk = lambda df, Y: TramDataset(df["text"], Y, tokenizer, max_length)
    train_loader = DataLoader(mk(train, Y_train), batch_size=batch_size, shuffle=True)
    dev_loader = DataLoader(mk(dev, Y_dev), batch_size=batch_size * 2)
    test_loader = DataLoader(mk(test, Y_test), batch_size=batch_size * 2)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_classes, problem_type="multi_label_classification").to(device)

    pos_weight = None
    if use_pos_weight:
        pos_weight = torch.tensor(compute_pos_weight(Y_train, cap=pos_weight_cap), dtype=torch.float, device=device)
        print(f"  pos_weight: min={pos_weight.min():.1f} max={pos_weight.max():.1f}")
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    total_steps = len(train_loader) * epochs
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * warmup_ratio), total_steps)

    best_f1, best_state, bad_epochs = -1.0, None, 0
    for epoch in range(1, epochs + 1):
        tr_loss, _, _ = run_epoch(model, train_loader, loss_fn, device, optimizer, scheduler, desc=f"Epoch {epoch} [train]")
        dv_loss, dv_probs, dv_labels = run_epoch(model, dev_loader, loss_fn, device, desc=f"Epoch {epoch} [dev]")
        m = evaluate(dv_labels, apply_thresholds(dv_probs, best_global_threshold(dv_labels, dv_probs)))
        print(f"  epoch {epoch}: train_loss={tr_loss:.4f} dev_loss={dv_loss:.4f} dev_micro_f1={m[\'micro_f1\']:.4f} dev_macro_f1={m[\'macro_f1\']:.4f}")
        if m["micro_f1"] > best_f1:
            best_f1, bad_epochs = m["micro_f1"], 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f"  early stop: no dev improvement for {patience} epochs")
                break
    model.load_state_dict(best_state)

    _, dv_probs, dv_labels = run_epoch(model, dev_loader, loss_fn, device, desc="Threshold tuning")
    thresholds = best_per_class_thresholds(dv_labels, dv_probs) if per_class_threshold else best_global_threshold(dv_labels, dv_probs)
    _, te_probs, te_labels = run_epoch(model, test_loader, loss_fn, device, desc="Test")
    te_preds = apply_thresholds(te_probs, thresholds)
    metrics = evaluate(te_labels, te_preds)
    print(f"  TEST: {{k: round(v, 4) if isinstance(v, float) else v for k, v in metrics.items()}}")

    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    np.save(f"{output_dir}/mlb_classes.npy", mlb.classes_)
    np.save(f"{output_dir}/test_probs.npy", te_probs)
    np.save(f"{output_dir}/test_preds.npy", te_preds)
    np.save(f"{output_dir}/test_labels.npy", te_labels)
    json.dump(np.asarray(thresholds).tolist(), open(f"{output_dir}/thresholds.json", "w"))
    json.dump({"model_name": model_name, "seed": seed, "dev_best_micro_f1": best_f1, "test": metrics, "config": dict(lr=lr, batch_size=batch_size, max_length=max_length, use_pos_weight=use_pos_weight, pos_weight_cap=pos_weight_cap, per_class_threshold=per_class_threshold)}, open(f"{output_dir}/metrics.json", "w"), indent=2)
    print(f"  saved to {output_dir}")
    return metrics

SyntaxError: unexpected character after line continuation character (3924054853.py, line 31)

**Reasoning**:
The training infrastructure was successfully registered in the previous cell. Now we define the main train_model function in the kernel to allow subsequent model training tasks.



In [5]:
def train_model(model_name, output_dir, seed=42, epochs=15, patience=3, batch_size=16, lr=2e-5, max_length=128, warmup_ratio=0.1, use_pos_weight=True, pos_weight_cap=20.0, per_class_threshold=False):
    set_seed(seed)
    os.makedirs(output_dir, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    num_classes = len(mlb.classes_)
    print(f"[{model_name} seed={seed}] device={device} train={len(train)} dev={len(dev)} test={len(test)} classes={num_classes}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    mk = lambda df, Y: TramDataset(df["text"], Y, tokenizer, max_length)
    train_loader = DataLoader(mk(train, Y_train), batch_size=batch_size, shuffle=True)
    dev_loader = DataLoader(mk(dev, Y_dev), batch_size=batch_size * 2)
    test_loader = DataLoader(mk(test, Y_test), batch_size=batch_size * 2)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_classes, problem_type="multi_label_classification").to(device)

    pos_weight = None
    if use_pos_weight:
        pos_weight = torch.tensor(compute_pos_weight(Y_train, cap=pos_weight_cap), dtype=torch.float, device=device)
        print(f"  pos_weight: min={pos_weight.min():.1f} max={pos_weight.max():.1f}")
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    total_steps = len(train_loader) * epochs
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * warmup_ratio), total_steps)

    best_f1, best_state, bad_epochs = -1.0, None, 0
    for epoch in range(1, epochs + 1):
        tr_loss, _, _ = run_epoch(model, train_loader, loss_fn, device, optimizer, scheduler, desc=f"Epoch {epoch} [train]")
        dv_loss, dv_probs, dv_labels = run_epoch(model, dev_loader, loss_fn, device, desc=f"Epoch {epoch} [dev]")
        m = evaluate(dv_labels, apply_thresholds(dv_probs, best_global_threshold(dv_labels, dv_probs)))
        print(f"  epoch {epoch}: train_loss={tr_loss:.4f} dev_loss={dv_loss:.4f} dev_micro_f1={m['micro_f1']:.4f} dev_macro_f1={m['macro_f1']:.4f}")
        if m["micro_f1"] > best_f1:
            best_f1, bad_epochs = m["micro_f1"], 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f"  early stop: no dev improvement for {patience} epochs")
                break
    model.load_state_dict(best_state)

    _, dv_probs, dv_labels = run_epoch(model, dev_loader, loss_fn, device, desc="Threshold tuning")
    thresholds = best_per_class_thresholds(dv_labels, dv_probs) if per_class_threshold else best_global_threshold(dv_labels, dv_probs)
    _, te_probs, te_labels = run_epoch(model, test_loader, loss_fn, device, desc="Test")
    te_preds = apply_thresholds(te_probs, thresholds)
    metrics = evaluate(te_labels, te_preds)
    print(f"  TEST: {{k: round(v, 4) if isinstance(v, float) else v for k, v in metrics.items()}}")

    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    np.save(f"{output_dir}/mlb_classes.npy", mlb.classes_)
    np.save(f"{output_dir}/test_probs.npy", te_probs)
    np.save(f"{output_dir}/test_preds.npy", te_preds)
    np.save(f"{output_dir}/test_labels.npy", te_labels)
    json.dump(np.asarray(thresholds).tolist(), open(f"{output_dir}/thresholds.json", "w"))
    json.dump({"model_name": model_name, "seed": seed, "dev_best_micro_f1": best_f1, "test": metrics, "config": dict(lr=lr, batch_size=batch_size, max_length=max_length, use_pos_weight=use_pos_weight, pos_weight_cap=pos_weight_cap, per_class_threshold=per_class_threshold)}, open(f"{output_dir}/metrics.json", "w"), indent=2)
    print(f"  saved to {output_dir}")
    return metrics

**Reasoning**:
The training infrastructure was successfully registered, and the warnings from MultiLabelBinarizer are expected since some evaluation labels do not exist in the training subset. Now, I will define the main `train_model` function in the kernel to allow subsequent model training tasks.



In [4]:
def train_model(model_name, output_dir, seed=42, epochs=15, patience=3, batch_size=16, lr=2e-5, max_length=128, warmup_ratio=0.1, use_pos_weight=True, pos_weight_cap=20.0, per_class_threshold=False):
    set_seed(seed)
    os.makedirs(output_dir, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    num_classes = len(mlb.classes_)
    print(f"[{model_name} seed={seed}] device={device} train={len(train)} dev={len(dev)} test={len(test)} classes={num_classes}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    mk = lambda df, Y: TramDataset(df["text"], Y, tokenizer, max_length)
    train_loader = DataLoader(mk(train, Y_train), batch_size=batch_size, shuffle=True)
    dev_loader = DataLoader(mk(dev, Y_dev), batch_size=batch_size * 2)
    test_loader = DataLoader(mk(test, Y_test), batch_size=batch_size * 2)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_classes, problem_type="multi_label_classification").to(device)

    pos_weight = None
    if use_pos_weight:
        pos_weight = torch.tensor(compute_pos_weight(Y_train, cap=pos_weight_cap), dtype=torch.float, device=device)
        print(f"  pos_weight: min={pos_weight.min():.1f} max={pos_weight.max():.1f}")
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    total_steps = len(train_loader) * epochs
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * warmup_ratio), total_steps)

    best_f1, best_state, bad_epochs = -1.0, None, 0
    for epoch in range(1, epochs + 1):
        tr_loss, _, _ = run_epoch(model, train_loader, loss_fn, device, optimizer, scheduler, desc=f"Epoch {epoch} [train]")
        dv_loss, dv_probs, dv_labels = run_epoch(model, dev_loader, loss_fn, device, desc=f"Epoch {epoch} [dev]")
        m = evaluate(dv_labels, apply_thresholds(dv_probs, best_global_threshold(dv_labels, dv_probs)))
        print(f"  epoch {epoch}: train_loss={tr_loss:.4f} dev_loss={dv_loss:.4f} dev_micro_f1={m['micro_f1']:.4f} dev_macro_f1={m['macro_f1']:.4f}")
        if m["micro_f1"] > best_f1:
            best_f1, bad_epochs = m["micro_f1"], 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f"  early stop: no dev improvement for {patience} epochs")
                break
    model.load_state_dict(best_state)

    _, dv_probs, dv_labels = run_epoch(model, dev_loader, loss_fn, device, desc="Threshold tuning")
    thresholds = best_per_class_thresholds(dv_labels, dv_probs) if per_class_threshold else best_global_threshold(dv_labels, dv_probs)
    _, te_probs, te_labels = run_epoch(model, test_loader, loss_fn, device, desc="Test")
    te_preds = apply_thresholds(te_probs, thresholds)
    metrics = evaluate(te_labels, te_preds)
    print(f"  TEST: {{k: round(v, 4) if isinstance(v, float) else v for k, v in metrics.items()}}")

    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    np.save(f"{output_dir}/mlb_classes.npy", mlb.classes_)
    np.save(f"{output_dir}/test_probs.npy", te_probs)
    np.save(f"{output_dir}/test_preds.npy", te_preds)
    np.save(f"{output_dir}/test_labels.npy", te_labels)
    json.dump(np.asarray(thresholds).tolist(), open(f"{output_dir}/thresholds.json", "w"))
    json.dump({"model_name": model_name, "seed": seed, "dev_best_micro_f1": best_f1, "test": metrics, "config": dict(lr=lr, batch_size=batch_size, max_length=max_length, use_pos_weight=use_pos_weight, pos_weight_cap=pos_weight_cap, per_class_threshold=per_class_threshold)}, open(f"{output_dir}/metrics.json", "w"), indent=2)
    print(f"  saved to {output_dir}")
    return metrics

**Reasoning**:
The previous code block failed due to backslash escapes inside the f-string (`m[\'micro_f1\']`). Python f-strings cannot contain backslashes inside their brace expressions in older Python versions, and escaping single quotes inside double-quoted f-strings is syntax-invalid. I will use standard single quotes without escaping since the outer string uses double quotes.



In [7]:
def train_model(model_name, output_dir, seed=42, epochs=15, patience=3, batch_size=16, lr=2e-5, max_length=128, warmup_ratio=0.1, use_pos_weight=True, pos_weight_cap=20.0, per_class_threshold=False):
    set_seed(seed)
    os.makedirs(output_dir, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    num_classes = len(mlb.classes_)
    print(f"[{model_name} seed={seed}] device={device} train={len(train)} dev={len(dev)} test={len(test)} classes={num_classes}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    mk = lambda df, Y: TramDataset(df["text"], Y, tokenizer, max_length)
    train_loader = DataLoader(mk(train, Y_train), batch_size=batch_size, shuffle=True)
    dev_loader = DataLoader(mk(dev, Y_dev), batch_size=batch_size * 2)
    test_loader = DataLoader(mk(test, Y_test), batch_size=batch_size * 2)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_classes, problem_type="multi_label_classification").to(device)

    pos_weight = None
    if use_pos_weight:
        pos_weight = torch.tensor(compute_pos_weight(Y_train, cap=pos_weight_cap), dtype=torch.float, device=device)
        print(f"  pos_weight: min={pos_weight.min():.1f} max={pos_weight.max():.1f}")
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    total_steps = len(train_loader) * epochs
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * warmup_ratio), total_steps)

    best_f1, best_state, bad_epochs = -1.0, None, 0
    for epoch in range(1, epochs + 1):
        tr_loss, _, _ = run_epoch(model, train_loader, loss_fn, device, optimizer, scheduler, desc=f"Epoch {epoch} [train]")
        dv_loss, dv_probs, dv_labels = run_epoch(model, dev_loader, loss_fn, device, desc=f"Epoch {epoch} [dev]")
        m = evaluate(dv_labels, apply_thresholds(dv_probs, best_global_threshold(dv_labels, dv_probs)))
        print(f"  epoch {epoch}: train_loss={tr_loss:.4f} dev_loss={dv_loss:.4f} dev_micro_f1={m['micro_f1']:.4f} dev_macro_f1={m['macro_f1']:.4f}")
        if m["micro_f1"] > best_f1:
            best_f1, bad_epochs = m["micro_f1"], 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f"  early stop: no dev improvement for {patience} epochs")
                break
    model.load_state_dict(best_state)

    _, dv_probs, dv_labels = run_epoch(model, dev_loader, loss_fn, device, desc="Threshold tuning")
    thresholds = best_per_class_thresholds(dv_labels, dv_probs) if per_class_threshold else best_global_threshold(dv_labels, dv_probs)
    _, te_probs, te_labels = run_epoch(model, test_loader, loss_fn, device, desc="Test")
    te_preds = apply_thresholds(te_probs, thresholds)
    metrics = evaluate(te_labels, te_preds)
    print(f"  TEST: {{k: round(v, 4) if isinstance(v, float) else v for k, v in metrics.items()}}")

    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    np.save(f"{output_dir}/mlb_classes.npy", mlb.classes_)
    np.save(f"{output_dir}/test_probs.npy", te_probs)
    np.save(f"{output_dir}/test_preds.npy", te_preds)
    np.save(f"{output_dir}/test_labels.npy", te_labels)
    json.dump(np.asarray(thresholds).tolist(), open(f"{output_dir}/thresholds.json", "w"))
    json.dump({"model_name": model_name, "seed": seed, "dev_best_micro_f1": best_f1, "test": metrics, "config": dict(lr=lr, batch_size=batch_size, max_length=max_length, use_pos_weight=use_pos_weight, pos_weight_cap=pos_weight_cap, per_class_threshold=per_class_threshold)}, open(f"{output_dir}/metrics.json", "w"), indent=2)
    print(f"  saved to {output_dir}")
    return metrics

## Train SecBERT Model

### Subtask:
Run the multi-seed model training loop on SecBERT and RoBERTa models across seeds 42, 43, and 44.


**Reasoning**:
Execute the training loop for the SecBERT and RoBERTa models across seeds 42, 43, and 44, skipping already completed runs to allow resume behavior.



In [9]:
MODELS = {"secbert": "jackaduma/SecBERT", "roberta": "roberta-base"}
SEEDS = [42, 43, 44]

for tag, name in MODELS.items():
    for s in SEEDS:
        out = f"{OUT_ROOT}/{tag}_s{s}"
        if os.path.exists(f"{out}/metrics.json"):
            print(f"skip {out} (already done)")
            continue
        print(f"\n>>> Training {tag} with seed {s}...")
        train_model(name, out, seed=s)


>>> Training secbert with seed 42...
[jackaduma/SecBERT seed=42] device=cuda train=3468 dev=604 test=725 classes=180


config.json:   0%|          | 0.00/467 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/378k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  336MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

... (320 outputs left out)

Test:   0%|          | 0/23 [00:00<?, ?it/s]

  TEST: {k: round(v, 4) if isinstance(v, float) else v for k, v in metrics.items()}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  saved to /content/drive/MyDrive/KDU/Thesis/runs/roberta_s44


**Reasoning**:
Verify the presence and validity of all model checkpoint metrics.json files on Google Drive for both models and all three seeds.



In [10]:
import os
import json

OUT_ROOT = "/content/drive/MyDrive/KDU/Thesis/runs"
MODELS = ["secbert", "roberta"]
SEEDS = [42, 43, 44]
required_keys = ["micro_f1", "macro_f1", "micro_precision", "micro_recall", "subset_accuracy"]

print("Verifying checkpoint artifacts...")
for model in MODELS:
    for seed in SEEDS:
        run_dir = f"{OUT_ROOT}/{model}_s{seed}"
        metrics_path = f"{run_dir}/metrics.json"
        if not os.path.exists(metrics_path):
            print(f"[-] MISSING: {metrics_path}")
            continue
        try:
            with open(metrics_path, 'r') as f:
                data = json.load(f)
            test_metrics = data.get("test", {})
            missing_keys = [k for k in required_keys if k not in test_metrics]
            if missing_keys:
                print(f"[!] INVALID: {metrics_path} (missing keys: {missing_keys})")
            else:
                print(f"[+] VALID: {metrics_path} (F1 Micro: {test_metrics['micro_f1']:.4f})")
        except Exception as e:
            print(f"[!] ERROR reading {metrics_path}: {e}")

Verifying checkpoint artifacts...
[+] VALID: /content/drive/MyDrive/KDU/Thesis/runs/secbert_s42/metrics.json (F1 Micro: 0.6377)
[+] VALID: /content/drive/MyDrive/KDU/Thesis/runs/secbert_s43/metrics.json (F1 Micro: 0.6311)
[+] VALID: /content/drive/MyDrive/KDU/Thesis/runs/secbert_s44/metrics.json (F1 Micro: 0.6400)
[+] VALID: /content/drive/MyDrive/KDU/Thesis/runs/roberta_s42/metrics.json (F1 Micro: 0.6149)
[+] VALID: /content/drive/MyDrive/KDU/Thesis/runs/roberta_s43/metrics.json (F1 Micro: 0.6237)
[+] VALID: /content/drive/MyDrive/KDU/Thesis/runs/roberta_s44/metrics.json (F1 Micro: 0.6252)


In [9]:
print("done")

# Task
Evaluate and verify the experimental metrics generated across all trained checkpoints for both the SecBERT (`jackaduma/SecBERT`) and RoBERTa (`roberta-base`) models trained over multiple seeds (42, 43, 44) under `"/content/drive/MyDrive/KDU/Thesis/runs"`. Extract, process, and compile the test evaluation results (including Micro-F1, Macro-F1, Precision, Recall, and Subset Accuracy) to construct and output a formatted, publishable summary results table showing the statistical mean and standard deviation across seeds.

## Verify Checkpoint Artifacts

### Subtask:
Verify the presence and integrity of all evaluation metrics and checkpoint files across the designated runs.


## Compile Statistical Results Table

### Subtask:
Load the evaluation metrics for each seed, compute the statistical mean and standard deviation for key metrics, and structure them into a summary DataFrame.


**Reasoning**:
Load the evaluation metrics for all seeds and models, calculate their mean and standard deviation, and compile them into a summary DataFrame.



In [11]:
import os
import json
import numpy as np
import pandas as pd

OUT_ROOT = "/content/drive/MyDrive/KDU/Thesis/runs"
MODELS = {"secbert": "SecBERT", "roberta": "RoBERTa"}
SEEDS = [42, 43, 44]
METRICS_TO_COLLECT = ["micro_f1", "macro_f1", "micro_precision", "micro_recall", "subset_accuracy"]

rows = []
for tag, name in MODELS.items():
    model_metrics = {m: [] for m in METRICS_TO_COLLECT}
    n_seeds = 0
    for s in SEEDS:
        metrics_path = f"{OUT_ROOT}/{tag}_s{s}/metrics.json"
        if os.path.exists(metrics_path):
            with open(metrics_path, 'r') as f:
                data = json.load(f)
            test_data = data.get("test", {})
            for m in METRICS_TO_COLLECT:
                if m in test_data:
                    model_metrics[m].append(test_data[m])
            n_seeds += 1

    if n_seeds > 0:
        row = {"Model": name, "Seeds": n_seeds}
        for m in METRICS_TO_COLLECT:
            vals = np.array(model_metrics[m])
            row[m] = f"{vals.mean():.3f} ± {vals.std():.3f}"
        rows.append(row)

results_df = pd.DataFrame(rows)
print("=== Summary Results Table ===")
print(results_df.to_string(index=False))
results_df

=== Summary Results Table ===
  Model  Seeds      micro_f1      macro_f1 micro_precision  micro_recall subset_accuracy
SecBERT      3 0.636 ± 0.004 0.390 ± 0.012   0.642 ± 0.020 0.631 ± 0.012   0.558 ± 0.026
RoBERTa      3 0.621 ± 0.005 0.364 ± 0.007   0.631 ± 0.007 0.612 ± 0.008   0.569 ± 0.006


,Model,Seeds,micro_f1,macro_f1,micro_precision,micro_recall,subset_accuracy
0,SecBERT,3,0.636 ± 0.004,0.390 ± 0.012,0.642 ± 0.020,0.631 ± 0.012,0.558 ± 0.026
1,RoBERTa,3,0.621 ± 0.005,0.364 ± 0.007,0.631 ± 0.007,0.612 ± 0.008,0.569 ± 0.006


## Final Task

### Subtask:
Present the finalized comparative results table highlighting performance differences across the trained models and seeds.


## Summary:

### Q&A

**Q: How do the SecBERT and RoBERTa models compare overall across the evaluated seeds?**

A: **SecBERT** outperformed **RoBERTa** on most classification metrics:
* **Micro-F1:** $0.636 \pm 0.004$ (SecBERT) vs. $0.621 \pm 0.005$ (RoBERTa)
* **Macro-F1:** $0.390 \pm 0.012$ (SecBERT) vs. $0.364 \pm 0.007$ (RoBERTa)
* **Micro-Precision:** $0.642 \pm 0.020$ (SecBERT) vs. $0.631 \pm 0.007$ (RoBERTa)
* **Micro-Recall:** $0.631 \pm 0.012$ (SecBERT) vs. $0.612 \pm 0.008$ (RoBERTa)

However, **RoBERTa** demonstrated slightly better exact match performance and stability:
* **Subset Accuracy:** $0.569 \pm 0.006$ (RoBERTa) vs. $0.558 \pm 0.026$ (SecBERT)

---

### Data Analysis Key Findings

* **SecBERT Edge in Classification Quality:** Domain-specific pre-training gives SecBERT a slight but consistent advantage in both Micro-F1 ($0.636$) and Macro-F1 ($0.390$) over the general-domain RoBERTa model (Micro-F1 of $0.621$ and Macro-F1 of $0.364$).
* **Low Macro-F1 Across Both Models:** Both models exhibited a significant gap between Micro-F1 (above $0.620$) and Macro-F1 (below $0.400$), indicating challenging class imbalances where rare classes are likely poorly predicted.
* **RoBERTa's High Stability:** RoBERTa showed significantly lower standard deviations across seeds, particularly for subset accuracy ($0.006$ vs. SecBERT's $0.026$) and Micro-Precision ($0.007$ vs. SecBERT's $0.020$), pointing to more stable convergence during training.

---

### Insights or Next Steps

* **Address Class Imbalance:** Given the low Macro-F1 scores relative to Micro-F1 for both models, apply class-weighting, focal loss, or data augmentation techniques targeting minority classes to improve performance on rare labels.
* **Explore Domain-Specific Hybrid Architectures:** Leverage SecBERT's stronger contextual representation with training stabilizers (such as layer-wise learning rate decay or adversarial training) to capture the performance edge of domain pre-training while matching the stability of RoBERTa.
